# 第73章 交互小提琴图（px.violin）

<!-- module-learning-arc:start -->
> **Plotly 模块主线｜第 10 / 18 步：交互观察分布与矩阵**
>
> **持续应用背景：** 准备周度经营预警会：让读者通过悬停、缩放、下钻和层级探索，沿着异常、定位、行动的路径完成追问。
>
> **承接上一阶段：** 交互箱线图（px.box）  →  **本章任务：** 交互小提琴图（px.violin）  →  **下一步：** 交互热力图（px.imshow）
>
> **大作业连接：** 本章练习将成为《周度经营预警会：交互诊断与行动看板》的一部分，最终需要把诊断和行动视图组织成支持经营预警决策的可分享 HTML。
<!-- module-learning-arc:end -->


## 本章场景

散点图靠点的位置回答问题，可当样本量很大时，几百上千个点叠在一起反而看不清数据到底集中在哪儿、偏不偏、有几个峰。


## 本章目标

学完本章，你将能够：

- **理解**：理解「交互小提琴图（px.violin）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「交互小提琴图（px.violin）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「交互小提琴图（px.violin）」并读出其中的结论。


## 适用场景
**背景引入**：散点图靠点的位置回答问题，可当样本量很大时，几百上千个点叠在一起反而看不清数据到底集中在哪儿、偏不偏、有几个峰。小提琴图把密度信息画成左右对称的"琴身"，同时保留按品类、渠道分组比较的能力，让你在一条坐标轴上就能看出不同组的分布长什么样。遇到样本量大、既要看分布形状又不想逐个箱子数的场景，它往往比箱线图更快帮你找到规律。（可以把它想成把直方图“转身再磨光滑”：某个数值区间的数据越多，琴身在那里就越鼓；它用左右对称的轮廓代替一根根柱子，样本一多也能一眼看出集中在哪、有几个峰。）

样本量充足，需要比较分布形状、偏态和多峰。


## 数据结构

分类列和连续数值列。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 box=True 改为 box=False，观察内部箱线摘要对中心信息展示的影响
2. 修改 points="outliers" 为 points="all"，对比仅显示异常点与显示全部原始点
3. 将 violinmode="group" 改为 "overlay"，说明分组模式对多组密度比较的作用


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `px.violin()`、`fig.update_layout()`、`fig.show()` | 样本量充足，需要比较分布形状、偏态和多峰。 | 小样本密度不稳定 |
| 进阶变体 | `px.violin()`、`fig.update_layout()`、`fig.show()` | 在基础图表上增加分组、注释、布局或交互 | 多组叠加遮挡 |
| 关键参数 | `box` | 箱线摘要 | 小样本密度不稳定 |
| 关键参数 | `points` | 原始点 | 多组叠加遮挡 |
| 关键参数 | `violinmode` | group/overlay | 用宽度比较绝对样本量 |
| 关键参数 | `spanmode` | 范围 | 小样本密度不稳定 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-73 -->
### 数学推导｜核密度估计把样本平滑成分布

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜每个样本放置一个核。** 以 $x_i$ 为中心、带宽为 $h$ 的核为

$$
K_h(x-x_i)=\frac{1}{h}K\!\left(\frac{x-x_i}{h}\right)
$$

$1/h$ 保证拉宽曲线后面积仍为 1。

**第 2 步｜把所有小曲线平均。** $n$ 个单位面积核相加后再除以 $n$，总面积仍为 1，于是得到 $\hat f_h(x)$。

**第 3 步｜理解带宽。** 较小 $h$ 保留局部起伏但方差大；较大 $h$ 更平滑但可能抹掉真实结构。

**把上面的关系收束为本章计算式：**

$$
\hat{f}_h(x)=\frac{1}{nh}\sum_{i=1}^{n}K\!\left(\frac{x-x_i}{h}\right)
$$

**符号解释：** $K$ 是核函数，$h$ 是带宽；$h$ 越大曲线越平滑。

**代码对应：** 调整 `bw_adjust`（或旧版带宽参数）并与原始样本/直方图交叉检查。

**使用边界：** KDE 会在观测范围外延伸，小样本或有自然边界的数据尤其要谨慎。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 1️⃣ 数据导入：手建两个小表 + 读取三个公开数据集
funnel = pd.DataFrame(
    {
        "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
        "users": [12000, 7200, 3100, 1850, 1420],
    }
)
timeline = pd.DataFrame(
    {
        "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
        "start": pd.to_datetime(
            ["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]
        ),
        "finish": pd.to_datetime(
            ["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]
        ),
        "owner": ["数据", "分析", "分析", "负责人"],
    }
)
diamonds = pd.read_csv("/datasets/diamonds.csv")
flights = pd.read_csv("/datasets/flights.csv")
gapminder = pd.read_csv("/datasets/gapminder.csv")
print(
    f'Diamonds {len(diamonds):,} | Flights {len(flights):,} | Gapminder {len(gapminder):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：为各章图表构造分析所需的派生字段
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"),
    category=diamonds["cut"],
    region=diamonds["clarity"],
    channel=diamonds["color"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    sales=diamonds["price"],
    month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55)

monthly = (
    flights.query("year == 1960")
    .rename(columns={"passengers": "sales"})
    .copy()
)
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()

regional = orders_full.groupby(["region", "channel"], as_index=False)[
    "sales"
].sum()

hierarchy = (
    diamonds.groupby(["cut", "color"], as_index=False)["price"]
    .sum()
    .rename(
        columns={"cut": "department", "color": "category", "price": "sales"}
    )
)

countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"],
    market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"],
    growth=lambda frame: frame["lifeExp"],
)
print(f"样本：orders {len(orders):,} | monthly {len(monthly):,} 行")


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.violin(
    orders,
    x="category",
    y="order_value",
    color="category",
    box=True,
    points=False,
    title="品类客单价小提琴图",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="客单价（元）", showlegend=False
)
fig.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**

基础图表里默认用 `box=True` 保留箱线摘要、`points=False` 隐藏原始点。请把 `points` 参数从 `False` 改成 `"all"`，再运行一次，观察原始点在每个品类的小提琴图上是怎么铺开的。想一想：显示全部点之后，哪个品类更能看出数据内部的疏密或离群情况，箱线摘要和原始点谁更适合判断中心？

在下面的代码格里补全代码，最后一行为画出修改后的图表。


In [ ]:
try:
    # 请在下方填写代码
    # 目标：把基础图表的 points 从 False 改成 "all"，显示全部原始点。
    points = False  # TODO: 改成 "all" 以显示全部原始点

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
fig = px.violin(
    orders,
    x="category",
    y="order_value",
    color="channel",
    box=True,
    points="outliers",
    violinmode="group",
    title="渠道与品类客单价密度",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="客单价（元）", legend_title="渠道"
)
fig.show()


## 参数说明

- box：箱线摘要
- points：原始点
- violinmode：group/overlay
- spanmode：范围


## 结果解读

宽度是估计密度，Hover可查看具体点；必须结合样本量解释。


## 本章实训：交互图与信息层次

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
_demo_fig = px.bar(report, x="region", y="sales", title="地区销售额")
_demo_fig.show()


### 第一个结果怎么读

Plotly 的基本流程是：准备表格、映射字段、设置标题、显示图形。悬停提示只能补充信息，不能替代坐标轴和单位。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
_demo_fig = px.bar(
    report.sort_values("sales", ascending=False),
    x="region",
    y="sales",
    text="sales",
    title="按销售额排序的地区销售额",
)
_demo_fig.update_traces(textposition="outside")
_demo_fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
_demo_fig.show()


### 第二个结果怎么读

第二个实验增加数值标签并排序。请检查：标签是否遮挡、标题是否准确、图形是否仍然能在窄屏阅读。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：空数据还能不能画图

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import plotly.express as px

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
if report.empty:
    print("没有可绘制的数据，请先检查筛选条件。")
else:
    fig = px.bar(report, x="region", y="sales", title="地区销售额")
    fig.update_layout(yaxis_title="销售额", xaxis_title="地区")
    fig.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

筛选后先判断是否为空，再调用绘图函数。空表不是绘图库的问题，而是上游筛选口径需要检查。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 小样本密度不稳定
- 多组叠加遮挡
- 用宽度比较绝对样本量


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

在默认图可读的前提下，增加一个 hover 字段或筛选交互。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：按渠道分面，观察不同渠道的密度形状
    # 【目标】换 x 分组变量，比较不同渠道的密度形状差异。
    import plotly.express as px

    # 起点示例(已可运行)：x 换成 channel，观察不同渠道密度。
    fig = px.violin(
        orders,
        x="channel",
        y="order_value",
        color="channel",
        box=True,
        points=False,
        title="分渠道客单价小提琴图",
    )
    fig.update_layout(
        xaxis_title="渠道", yaxis_title="客单价（元）", showlegend=False
    )
    fig.show()

    # ---- 反思记录：不同渠道的密度形状(胖瘦/峰)有何不同 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用交互小提琴图展示平滑密度，并可同时显示箱线和原始点。


### 你已经掌握

- 判断交互小提琴图（px.violin）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `box` | 箱线摘要 |
| `points` | 原始点 |
| `violinmode` | group/overlay |
| `spanmode` | 范围 |


### 需要注意

- 小样本密度不稳定
- 多组叠加遮挡
- 用宽度比较绝对样本量


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


In [ ]:
# 请在下方填写代码（参考答案）

points = "all"
fig = px.violin(
    orders,
    x="category",
    y="order_value",
    color="category",
    box=True,
    points=points,
    title="品类客单价小提琴图（显示原始点）",
)
fig.update_layout(
    xaxis_title="品类", yaxis_title="客单价（元）", showlegend=False
)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
fig = px.violin(
    orders,
    x="region",
    y="items",
    color="region",
    box=True,
    points="all",
    title="区域购买件数分布",
)
fig.update_traces(jitter=0.15, marker_size=3)
fig.update_layout(xaxis_title="区域", yaxis_title="购买件数", showlegend=False)
fig.show()
